In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import re

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.28 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
# reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation
reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

In [4]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = {}
for citation, text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()):
    # if citation in court_consideration_d:
    #     court_consideration_d[citation] = court_consideration_d[citation] + '\n\n' + text
    # else:
    #     court_consideration_d[citation] = text
    court_consideration_d[citation] = text

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_001.csv')

_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}
    

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded")

data loaded


In [5]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_dense_index.info()

True
DenseIndex.embeddings:  (2107648, 1024)
[dense_index] documents.len: 1985178 parent_idx.len: 2107648


In [6]:
from sparse_index import SparseIndex

court_sparse_index = SparseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_sparse_index.load()

In [ ]:
import citation_utils
import rerank_utils

court_topk=1000

id_l = []
all_hits_l = []
test_df = pd.read_csv("../data/test_rewrite_001.csv")

for id, query in tqdm(zip(test_df['query_id'].tolist(), 
                                      test_df['query'].tolist()), 
                                  total=len(test_df), 
                                  desc="test-data") :

    court_recall = court_dense_index.search_with_score(query, top_k=court_topk)

    reranked_court = rerank_utils.rerank_by_dense_batch_chunked(reranker, query, [hit for hit,score in court_recall], len(court_recall), 10, 384, 128)

    print("reranked_court.len:", len(reranked_court))
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(reranked_court, decay="reciprocal")[:100]

    print(second_layer[0])
    all_hits = []
    for citation, score in second_layer:
        # print('citation:', citation)
        if citation in court_consideration_d:
            all_hits.append({'citation':citation, 'text':court_consideration_d[citation]})
        elif citation in law_d:
            all_hits.append({'citation':citation, 'text':law_d[citation]})
            
    all_hits_l.append(all_hits)
    
    print("second_layer.len:", len(second_layer), "all_hits.len:", len(all_hits))
    
print(len(all_hits_l), len(all_hits_l[0]))

test-data:   0%|          | 0/40 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


reranked_court.len: 946


test-data:   2%|▎         | 1/40 [01:07<43:46, 67.36s/it]

('Art. 3 Abs. 1 lit', 5.472248052245603)
second_layer.len: 100 all_hits.len: 27
reranked_court.len: 985


test-data:   5%|▌         | 2/40 [02:15<43:05, 68.05s/it]

...

('Art. 5 Abs. 2 AHVG', 3.0826387521297898)
second_layer.len: 100 all_hits.len: 45


test-data:  35%|███▌      | 14/40 [14:07<22:40, 52.33s/it]

reranked_court.len: 977
('Art. 6 Abs. 1 UVG', 237.77000396266206)
second_layer.len: 100 all_hits.len: 33
reranked_court.len: 915


test-data:  38%|███▊      | 15/40 [15:21<24:37, 59.10s/it]

('Art. 125 Abs. 1 ZGB', 14.985296978513004)
second_layer.len: 100 all_hits.len: 41


In [ ]:
id_l = test_df['query_id'].tolist()
predicted_citations_l = []
print(len(all_hits_l))
for all_hits in all_hits_l:
    predicted_citations_l.append(';'.join(list(set([hit['citation'] for hit in all_hits[:35]]))))
print(len(predicted_citations_l))
result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':predicted_citations_l})
result_df.to_csv("../data/submission.csv", index=False)